# 25. ?????????????? (Advanced Categorical Variable Encoding)

## ?? ??????

| ?? | ?? |
|------|------|
| ?? ?????? | 90-120? |
| ?? ??? | ?????(?????) |
| ?? ???? | ??????????????? |
| ?? ??????? | category_encoders, scikit-learn, Pandas |

### ????
- Pandas?????
- ???????(?????)
- ???????

---

## ??
????????????????????????????????Target Encoding?Frequency Encoding?Embedding??????????????????????

## ????
- ?????????????????
- ??????????????????
- Target Encoding?????????
- ???????????????
- Embedding?????????
- ???????????????????

---

## ?? ??????????????????????????

### ?????????????

??????????????**???????**?????:

```python
# ? ??????
model.fit(["Tokyo", "Osaka", "Nagoya"], y)

# ? ??????
model.fit([0, 1, 2], y)  # ??? [[1,0,0], [0,1,0], [0,0,1]]
```

### ???????????

| ?? | ?? |
|------|------|
| ??? | ????????????????? |
| ???? | ?????????????? |
| ???? | ????????????? |
| ????? | ?????????????? |

### ?????????????

```
??????????           ??????
?(2-10)                 ?  One-Hot Encoding
                              ????????
                              
?(10-50)                ?  Target Encoding / WoE
                              ??????????
                              
?(50+)                  ?  Target Encoding / Embedding
                              ???????
```

### ?? ??????????

| NG | ?? | OK |
|----|------|-----|
| ??????????One-Hot | ????(???????) | Target Encoding |
| Target Encoding???? | ??????????? | Out-of-Fold?? |
| ???????Label | ????????????? | Ordinal Encoding |

In [ ]:
# ??????????????
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from category_encoders import TargetEncoder, LeaveOneOutEncoder, WOEEncoder, CatBoostEncoder
import warnings
warnings.filterwarnings('ignore')

# ??
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)

# ?????????
import matplotlib.font_manager as fm

def setup_japanese_font():
    """????????????"""
    japanese_fonts = [
        'Hiragino Sans', 'Hiragino Maru Gothic Pro', 'AppleGothic',  # macOS
        'Yu Gothic', 'MS Gothic',  # Windows
        'Noto Sans CJK JP', 'IPAexGothic', 'TakaoPGothic',  # Linux
    ]
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()
if font_used:
    print(f"Japanese font: {font_used}")


## 1. ??????????

### ???????????

**????????(Categorical Variable)**???????(????)????????

### ???????????

1. **????(Nominal)**
   - ???????
   - ?: ?(?????)???????????

2. **????(Ordinal)**
   - ???????
   - ?: ?????(??? < ??? < ?? < ??)????(S < M < L < XL)

3. **????(Binary)**
   - 2?????
   - ?: ??(?/?)?Yes/No

### ??????????????

??????????????**?????**????????

- ? ?????????(Random Forest?XGBoost??): ???????????????
- ? ?????(Logistic Regression?SVM??): ??????

### ??????

- ?? **???**: ????????
- ?? **E????**: ???????????
- ?? **??**: ???????
- ?? **??**: ???????

## 2. ??????????

In [ ]:
# ??????????(?????????)
np.random.seed(42)
n_samples = 5000

# ????????
cities = ['Tokyo', 'Osaka', 'Nagoya', 'Fukuoka', 'Sapporo', 'Sendai', 'Hiroshima', 'Kyoto']
occupations = ['Engineer', 'Doctor', 'Teacher', 'Sales', 'Manager', 'Student', 'Self-employed', 
               'Artist', 'Unemployed', 'Retired']
education = ['High School', 'Bachelor', 'Master', 'PhD']
property_type = ['Apartment', 'House', 'Condo', 'Townhouse']
marital_status = ['Single', 'Married', 'Divorced', 'Widowed']

# ?????
data = {
    'city': np.random.choice(cities, n_samples),
    'occupation': np.random.choice(occupations, n_samples),
    'education': np.random.choice(education, n_samples),
    'property_type': np.random.choice(property_type, n_samples),
    'marital_status': np.random.choice(marital_status, n_samples),
    'age': np.random.randint(22, 70, n_samples),
    'income': np.random.lognormal(13, 0.5, n_samples),  # ??(??)
    'loan_amount': np.random.lognormal(15, 0.4, n_samples)  # ????(??)
}

df = pd.DataFrame(data)

# ???????(??/??)???
# ??????????????????????????????
approval_score = (
    (df['income'] / df['income'].mean()) * 0.3 +
    (df['education'].map({'High School': 0.5, 'Bachelor': 0.7, 'Master': 0.85, 'PhD': 1.0})) * 0.25 +
    (df['occupation'].map({
        'Doctor': 1.0, 'Engineer': 0.9, 'Manager': 0.85, 'Teacher': 0.75,
        'Sales': 0.6, 'Self-employed': 0.55, 'Artist': 0.5, 
        'Student': 0.3, 'Unemployed': 0.1, 'Retired': 0.4
    })) * 0.25 +
    (1 - df['loan_amount'] / df['income']) * 0.2 +
    np.random.normal(0, 0.1, n_samples)
)

df['approved'] = (approval_score > approval_score.median()).astype(int)

print(f"??????: {df.shape}")
print(f"\n??????????: {df.select_dtypes(include='object').shape[1]}")
print(f"??????: {df.select_dtypes(include=['int64', 'float64']).shape[1] - 1}")
print(f"\n???????:")
print(df['approved'].value_counts())
print(f"\n??????:")
print(df.head(10))

In [ ]:
# ???????????
categorical_cols = df.select_dtypes(include='object').columns.tolist()

print("???????????:")
print("=" * 60)

for col in categorical_cols:
    n_unique = df[col].nunique()
    print(f"\n{col}:")
    print(f"  ??????: {n_unique}")
    print(f"  ?: {df[col].unique()[:10].tolist()}")
    
    # ????????(??????)??????
    if n_unique == 2:
        cardinality = "?(Binary)"
    elif n_unique <= 5:
        cardinality = "?"
    elif n_unique <= 10:
        cardinality = "?"
    else:
        cardinality = "?"
    
    print(f"  ????????: {cardinality}")

## 3. ??????????????

### 3.1 Label Encoding

????????????????

- ? **????**: ????
- ? **??**: ???????????????????????

In [ ]:
# Label Encoding
from sklearn.preprocessing import LabelEncoder

df_label = df.copy()

# ?????(????)
education_order = {'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3}
df_label['education_label'] = df_label['education'].map(education_order)

print("Label Encoding(?????):")
print(df_label[['education', 'education_label']].drop_duplicates().sort_values('education_label'))

print("\n?? ????:")
print("- ????????(High School < Bachelor < Master < PhD)")
print("- ?????????(?????????????)")

### 3.2 One-Hot Encoding

??????????(?????)???????

- ? **????**: ??????????????
- ? **??**: ???????????????

In [ ]:
# One-Hot Encoding
df_onehot = df.copy()

# property_type(4????)?One-Hot Encoding
onehot_encoded = pd.get_dummies(df_onehot['property_type'], prefix='property')
df_onehot = pd.concat([df_onehot, onehot_encoded], axis=1)

print("One-Hot Encoding(?????):")
print(df_onehot[['property_type'] + onehot_encoded.columns.tolist()].head(10))

print(f"\n????: 1")
print(f"One-Hot????: {len(onehot_encoded.columns)}")

print("\n?? ????:")
print("- ??????????????")
print("- ????????")
print("- ??????????????????????")

In [ ]:
# ?????????????One-Hot Encoding???
print("?????????????(occupation: 10????):")
occupation_onehot = pd.get_dummies(df['occupation'], prefix='occ')
print(f"??: {len(occupation_onehot.columns)}")
print(f"\n?????????100?? ? 100?")
print(f"?????????1000?? ? 1000?(?????)")

print("\n?? ???:")
print("- ?????????")
print("- ???????")
print("- ???????")
print("- ????(?)??????")

## 4. ?????????????

### 4.1 Frequency Encoding(??????????)

????????????????????

In [ ]:
# Frequency Encoding
def frequency_encoding(df, column):
    """
    ??????????
    
    Parameters:
    -----------
    df : DataFrame
    column : str
        ???????
    
    Returns:
    --------
    encoded : Series
        ??????????
    """
    freq = df[column].value_counts(normalize=True)
    return df[column].map(freq)

df_freq = df.copy()
df_freq['city_freq'] = frequency_encoding(df_freq, 'city')
df_freq['occupation_freq'] = frequency_encoding(df_freq, 'occupation')

print("Frequency Encoding:")
print(df_freq[['city', 'city_freq', 'occupation', 'occupation_freq']].head(10))

# ?????
city_freq_summary = df_freq.groupby('city')['city_freq'].first().sort_values(ascending=False)
print("\n??????:")
print(city_freq_summary)

print("\n?? ????:")
print("- ???????(??????)")
print("- ??????????????")
print("- ?????")

print("\n?? ?????:")
print("- ?????????????????")
print("- ??????????")

### 4.2 Target Encoding(?????????????)

### ?? Target Encoding???

??????????????????????????

#### ?????

$$
\text{TargetEnc}(c) = \frac{n_c \cdot \bar{y}_c + m \cdot \bar{y}}{n_c + m}
$$

- $c$: ????
- $n_c$: ????c??????
- $\bar{y}_c$: ????c????????
- $\bar{y}$: ??????????
- $m$: ???????????

#### ??????

Target Encoding?**??????????????????**???:

```
??: ?????????
????     | ????    | Target Encoding
------------|------------|----------------
Doctor      | 1500??   | 1500
Engineer    | 800??    | 800
Student     | 100??    | 100
```

? ?????Doctor??????????????????

#### ?? ??: ?????????

**??**: ?????????**??????**???????

```
????????(????)?
1. ??????????????????????
2. ???????????
? ????????????????!

??????(Out-of-Fold)?
1. ??????K-Fold???
2. ?Fold????Fold?????????????
3. ????????????????????
? ?????
```

#### ??????????

????????????????????:

| ???? | ????? | ?? | ??? |
|---------|-----------|-----|--------|
| Doctor | 1000 | 0.8 | ? |
| Pilot | 3 | 1.0 | ?(????)|

????????????????????????????????

In [ ]:
# ?????Target Encoding(???????? - ???)
def naive_target_encoding(df, column, target):
    """
    ?????Target Encoding(??????!????????)
    """
    target_mean = df.groupby(column)[target].mean()
    return df[column].map(target_mean)

df_target_naive = df.copy()
df_target_naive['occupation_target_naive'] = naive_target_encoding(
    df_target_naive, 'occupation', 'approved'
)

print("?????Target Encoding(??):")
occupation_target = df_target_naive.groupby('occupation').agg({
    'approved': 'mean',
    'occupation_target_naive': 'first'
}).sort_values('approved', ascending=False)
print(occupation_target)

print("\n?? ??:")
print("- ????????????????")
print("- ???????????????????(??????)")
print("- ?????????")
print("- ???????????!")

In [ ]:
# ???Target Encoding(Out-of-Fold??)
from category_encoders import TargetEncoder

# ?????
X = df[categorical_cols + ['age', 'income', 'loan_amount']]
y = df['approved']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Target Encoder(category_encoders?????)
target_encoder = TargetEncoder(cols=categorical_cols, smoothing=1.0)

# ??????fit????
X_train_encoded = target_encoder.fit_transform(X_train, y_train)

# ???????????????????
X_test_encoded = target_encoder.transform(X_test)

print("???Target Encoding:")
print("\n?????(??????):")
print(X_train_encoded.head(10))

print("\n?? ???????:")
print("- fit()???????????")
print("- ????????transform()????")
print("- ??????????????")
print("- ?????????????Fold????fit")

In [ ]:
# Smoothing????????
print("Target Encoding???????:")
print("=" * 60)

# ??????
occupation_stats = df.groupby('occupation').agg({
    'approved': ['mean', 'count']
}).round(4)
occupation_stats.columns = ['approval_rate', 'count']
occupation_stats = occupation_stats.sort_values('count')

print(occupation_stats)

print("\n?? ??????????:")
print("- ????????????????????")
print("- ????????????????")
print("- ????????")
print("\n?: (count * mean + smoothing * global_mean) / (count + smoothing)")

### 4.3 CatBoost Encoding

CatBoost???????????????????Target Encoding???

In [ ]:
# CatBoost Encoding
from category_encoders import CatBoostEncoder

catboost_encoder = CatBoostEncoder(cols=categorical_cols)

X_train_catboost = catboost_encoder.fit_transform(X_train, y_train)
X_test_catboost = catboost_encoder.transform(X_test)

print("CatBoost Encoding:")
print("\n?????(??????):")
print(X_train_catboost.head(10))

print("\n?? CatBoost Encoding???:")
print("- Target Encoding????")
print("- ???????????????????")
print("- CatBoost?????????")
print("- ??????????")

### 4.4 Weight of Evidence (WoE) Encoding

**WoE Encoding**?????????????????????????????????

In [ ]:
# WoE Encoding
from category_encoders import WOEEncoder

woe_encoder = WOEEncoder(cols=categorical_cols)

X_train_woe = woe_encoder.fit_transform(X_train, y_train)
X_test_woe = woe_encoder.transform(X_test)

print("WoE Encoding:")
print("\n?????(??????):")
print(X_train_woe.head(10))

# ????WoE?
occupation_woe = X_train_woe.groupby(X_train['occupation'])['occupation'].first().sort_values()
print("\n????WoE?:")
print(occupation_woe)

print("\n?? WoE???:")
print("- ???: ???????????")
print("- ???: ???????????")
print("- 0???: ????????")
print("\n?: ln(P(approved|category) / P(rejected|category))")

## 5. ??????????????

### ???????????

???????????(????????)?????????????

In [ ]:
# ????????????????????(????)
np.random.seed(42)
prefectures = [f'Prefecture_{i:02d}' for i in range(1, 48)]  # 47????

# Zipf??(????????????)
zipf_probs = 1.0 / np.arange(1, 48)
zipf_probs = zipf_probs / zipf_probs.sum()

df_high_card = df.copy()
df_high_card['prefecture'] = np.random.choice(prefectures, n_samples, p=zipf_probs)

print(f"??????????: {df_high_card['prefecture'].nunique()}")
print("\n??10???????:")
print(df_high_card['prefecture'].value_counts().head(10))
print("\n??10???????:")
print(df_high_card['prefecture'].value_counts().tail(10))

In [ ]:
# ??1: ??????????????
def group_rare_categories(df, column, threshold=0.01, rare_label='Other'):
    """
    ??????????'Other'?????
    
    Parameters:
    -----------
    threshold : float
        ????????????????
    """
    freq = df[column].value_counts(normalize=True)
    rare_categories = freq[freq < threshold].index
    
    df_new = df.copy()
    df_new[column] = df_new[column].apply(
        lambda x: rare_label if x in rare_categories else x
    )
    
    return df_new, rare_categories

df_grouped, rare_prefs = group_rare_categories(df_high_card, 'prefecture', threshold=0.01)

print(f"???????: {df_high_card['prefecture'].nunique()}")
print(f"????????????: {df_grouped['prefecture'].nunique()}")
print(f"'Other'????????????: {len(rare_prefs)}")
print(f"\n?????????:")
print(df_grouped['prefecture'].value_counts().head(10))

In [ ]:
# ??2: Target Encoding(????????????)
# ??????? - Target Encoding??????????????

# ??3: Frequency Encoding
df_high_card['prefecture_freq'] = frequency_encoding(df_high_card, 'prefecture')

print("????????????????:")
print("=" * 60)
print("\n1. ????????????")
print("   - ????: ???????")
print("   - ?????: ?????")

print("\n2. Target Encoding")
print("   - ????: ??????????????")
print("   - ?????: ??????????")

print("\n3. Frequency Encoding")
print("   - ????: ???????")
print("   - ?????: ??????????")

print("\n4. Embedding(?????????)")
print("   - ????: ?????????")
print("   - ?????: ??????")

## 6. ?????????????

In [ ]:
# ??????????????????????
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

# ????????
numeric_cols = ['age', 'income', 'loan_amount']

results = []

# 1. Label Encoding(???????)
X_label = X.copy()
X_label['education'] = X_label['education'].map(education_order)
for col in categorical_cols:
    if col != 'education':
        le = LabelEncoder()
        X_label[col] = le.fit_transform(X_label[col])

X_train_label, X_test_label, y_train_label, y_test_label = train_test_split(
    X_label, y, test_size=0.2, random_state=42, stratify=y
)

rf_label = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_label.fit(X_train_label, y_train_label)
pred_label = rf_label.predict_proba(X_test_label)[:, 1]

results.append({
    'Encoding': 'Label Encoding',
    'Accuracy': accuracy_score(y_test_label, rf_label.predict(X_test_label)),
    'ROC-AUC': roc_auc_score(y_test_label, pred_label)
})

# 2. One-Hot Encoding
X_onehot = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

X_train_onehot, X_test_onehot, y_train_onehot, y_test_onehot = train_test_split(
    X_onehot, y, test_size=0.2, random_state=42, stratify=y
)

rf_onehot = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_onehot.fit(X_train_onehot, y_train_onehot)
pred_onehot = rf_onehot.predict_proba(X_test_onehot)[:, 1]

results.append({
    'Encoding': 'One-Hot Encoding',
    'Accuracy': accuracy_score(y_test_onehot, rf_onehot.predict(X_test_onehot)),
    'ROC-AUC': roc_auc_score(y_test_onehot, pred_onehot)
})

# 3. Target Encoding
rf_target = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_target.fit(X_train_encoded, y_train)
pred_target = rf_target.predict_proba(X_test_encoded)[:, 1]

results.append({
    'Encoding': 'Target Encoding',
    'Accuracy': accuracy_score(y_test, rf_target.predict(X_test_encoded)),
    'ROC-AUC': roc_auc_score(y_test, pred_target)
})

# 4. CatBoost Encoding
rf_catboost = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_catboost.fit(X_train_catboost, y_train)
pred_catboost = rf_catboost.predict_proba(X_test_catboost)[:, 1]

results.append({
    'Encoding': 'CatBoost Encoding',
    'Accuracy': accuracy_score(y_test, rf_catboost.predict(X_test_catboost)),
    'ROC-AUC': roc_auc_score(y_test, pred_catboost)
})

# 5. WoE Encoding
rf_woe = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_woe.fit(X_train_woe, y_train)
pred_woe = rf_woe.predict_proba(X_test_woe)[:, 1]

results.append({
    'Encoding': 'WoE Encoding',
    'Accuracy': accuracy_score(y_test, rf_woe.predict(X_test_woe)),
    'ROC-AUC': roc_auc_score(y_test, pred_woe)
})

# ?????
results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)

print("\n" + "=" * 70)
print("???????????????(Random Forest)")
print("=" * 70)
print(results_df.to_string(index=False))

# ?????
best_idx = results_df['ROC-AUC'].idxmax()
print(f"\n?? ?????: {results_df.loc[best_idx, 'Encoding']}")
print(f"   ROC-AUC: {results_df.loc[best_idx, 'ROC-AUC']:.4f}")

In [ ]:
# ??????
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].barh(results_df['Encoding'], results_df['Accuracy'], 
             alpha=0.7, edgecolor='black', color='skyblue')
axes[0].set_xlabel('Accuracy', fontsize=11)
axes[0].set_title('Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)
for i, (idx, row) in enumerate(results_df.iterrows()):
    axes[0].text(row['Accuracy'], i, f"{row['Accuracy']:.4f}", 
                va='center', ha='left', fontsize=9)

# ROC-AUC
axes[1].barh(results_df['Encoding'], results_df['ROC-AUC'], 
             alpha=0.7, edgecolor='black', color='salmon')
axes[1].set_xlabel('ROC-AUC', fontsize=11)
axes[1].set_title('ROC-AUC Comparison', fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)
for i, (idx, row) in enumerate(results_df.iterrows()):
    axes[1].text(row['ROC-AUC'], i, f"{row['ROC-AUC']:.4f}", 
                va='center', ha='left', fontsize=9)

plt.tight_layout()
plt.show()

## 7. ?????????????

### ????????????????

| ?? | ???? | ?? |
|------|---------|------|
| ?????????(<10) | One-Hot Encoding | ???????? |
| ?????????(10-50) | Target Encoding / WoE | ?????????? |
| ?????????(>50) | Target/CatBoost Encoding | ??????? |
| ???? | Ordinal Encoding | ??????? |
| ????????? | Label/Target Encoding | ?????? |
| ????? | One-Hot / WoE | ??????? |
| ????????? | Embedding | ????????? |

### ???????

```python
# ? ???????????????? ???????

# 1. ??????
# - ???????????????
# - ???????????????
# - ????????
# - ?????????

# 2. ?????????????
# - ????????????????????
# - ???????????????
# - ???????????

# 3. ?????????
# - Target Encoding?fit/transform??????
# - ???????????????????????
# - ????????????Fold???fit????

# 4. ???????????
# - ??????????????
# - Target Encoding???????
# - Embedding???????

# 5. ??
# - ????????????????????
# - ???????????????
# - ?????????????
```

### ????????

1. **??????**: Target Encoding?????????????
2. **?????**: ??????????One-Hot Encoding
3. **?????**: ?????Label Encoding?????
4. **???**: ?????????Target Encoding
5. **??????**: ???????????????

## 8. ???

### ?????????????

1. **???????????**
   - ??????????????
   - ???????????
   - ????????????

2. **????????????**
   - Label Encoding: ??????
   - One-Hot Encoding: ???????????
   - Ordinal Encoding: ?????

3. **???????????**
   - Frequency Encoding: ???????
   - Target Encoding: ????????????
   - CatBoost Encoding: Target Encoding????
   - WoE Encoding: ???????????

4. **???????????**
   - ?????????????
   - Target Encoding
   - Frequency Encoding
   - Embedding(?????????)

5. **?????????**
   - Out-of-Fold Target Encoding
   - fit/transform???
   - ??????????????

---

## ?? ??????

```python
# 1. One-Hot Encoding(?????????)
df_encoded = pd.get_dummies(df, columns=['category'], drop_first=True)

# 2. Label/Ordinal Encoding(????)
order_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['category_encoded'] = df['category'].map(order_map)

# 3. Frequency Encoding
freq = df['category'].value_counts(normalize=True)
df['category_freq'] = df['category'].map(freq)

# 4. Target Encoding(category_encoders??)
from category_encoders import TargetEncoder
encoder = TargetEncoder(cols=['category'])
X_train_enc = encoder.fit_transform(X_train, y_train)
X_test_enc = encoder.transform(X_test)  # fit??!

# 5. ????????????
freq = df['category'].value_counts(normalize=True)
rare = freq[freq < 0.01].index
df['category'] = df['category'].replace(rare, 'Other')

# 6. LightGBM/CatBoost??????????
lgb.LGBMClassifier(categorical_feature=['category'])
catboost.CatBoostClassifier(cat_features=['category'])
```

### ???????

- ? **???????????**: ???One-Hot????Target Encoding
- ? **????????**: Target Encoding?fit/transform???
- ? **?????**: ??????????
- ? **????**: 1??????????
- ? **?????**: ????????????????????
- ? **???????**: ??????????????????

---

## ? ???????????

### ???1: Target Encoding???????

```python
# ? ????:??????????
target_mean = df.groupby('category')['target'].mean()
df['category_enc'] = df['category'].map(target_mean)
X_train, X_test = split(df)  # ????????????

# ? ????:fit/transform???
from category_encoders import TargetEncoder
encoder = TargetEncoder(cols=['category'])
X_train_enc = encoder.fit_transform(X_train, y_train)  # ?????fit
X_test_enc = encoder.transform(X_test)  # transform??
```

**??**: CV???????????????????
**??**: ???????????????????????

---

### ???2: ??????????One-Hot Encoding

```python
# ? ????:1000?????One-Hot
df_encoded = pd.get_dummies(df, columns=['user_id'])  # 1000???!

# ? ????:Target Encoding???
from category_encoders import TargetEncoder
encoder = TargetEncoder(cols=['user_id'])
df_encoded = encoder.fit_transform(df, y)  # 1????
```

**??**: ????????????????
**??**: ????????????????????????

---

### ???3: ???????????????

```python
# ? ????:????????????????
encoder = LabelEncoder()
encoder.fit(train['category'])
test['category_enc'] = encoder.transform(test['category'])  # ???!

# ? ????:??????????
from category_encoders import TargetEncoder
encoder = TargetEncoder(cols=['category'], handle_unknown='value')
# ?????????????????
```

**??**: ???KeyError?ValueError
**??**: ?????????????????????

---

## ?? ???????

### Q1: One-Hot Encoding?????
<details>
<summary>?????</summary>

1. **?????**: ????????????(1000?????1000?)
2. **?????**: ???????0????????????
3. **??????**: ???????????????
4. **??????????**: 1?????????????????????
</details>

### Q2: Target Encoding???????????????
<details>
<summary>?????</summary>

**??????????????????????**

?: ???????3??????????????????
- ????????: 1.0(??)
- ????????: ????????(??????????)

??: $\text{enc} = \frac{n \cdot \bar{y}_{category} + m \cdot \bar{y}_{global}}{n + m}$
</details>

### Q3: Frequency Encoding????????
<details>
<summary>?????</summary>

1. **????????????**: ????????????????
2. **????????????**: ?????????????????
3. **?????**: ???????????????

????(????????????)??????????????????
</details>

### Q4: LightGBM/CatBoost????????????????????
<details>
<summary>?????</summary>

**?????????????????????**

- **LightGBM**: ???????????????????????
- **CatBoost**: Ordered Target Statistics??????Target Encoding???

?????:
1. One-Hot???????
2. ?????????Target Encoding
3. ??????
</details>

### Q5: ????????????????????????????
<details>
<summary>?????</summary>

**?????????????????**

| ????? | ? | ???? | ???? |
|-----------|-----|---------|---------|
| ???? | ???(S<M<L) | ?? | Ordinal Encoding |
| ???? | ?(?,?,?) | ?? | One-Hot / Target |

?????Label Encoding(0,1,2)??????????S<M<L?????????????
???????????????<?<??????????????????
</details>

---

## ??? ????

### ??1: ????Target Encoding???
???????????????Target Encoding?????????????

```python
def custom_target_encoding(train, test, column, target, smoothing=1.0):
    """
    ????Target Encoding
    
    Parameters:
    -----------
    smoothing : float
        ???????????(?????????????)
    """
    # ????????
    pass
```

### ??2: ???????????????
Kaggle??House Prices??????????????????????????:
- One-Hot Encoding
- Target Encoding
- Frequency Encoding
- LightGBM??????????

### ??3: ??????????
?????????????????????????????????????????:
- ??????????
- ????????????????
- ????????

---

## ? ?????????

- [ ] ???????????(????????)??????
- [ ] ????????????????????????????
- [ ] One-Hot Encoding??????????????
- [ ] Target Encoding?????????(????????)
- [ ] ??????????????????
- [ ] ?????????????????????
- [ ] LightGBM/CatBoost???????????????

---

### ??????

- Phase 8??!Phase 9(Kaggle?????)???
- ???Kaggle???????????????
- Embedding???????????????????
- ?????????????????????